# Diffusion-StyleGAN2 Own-Code Implementation for CIFAR-10

This notebook implements the Diffusion-GAN paper in PyTorch without cloning or importing the official repository.

Implementation target:

- Backbone: StyleGAN2-style generator and discriminator implemented from scratch
- Diffusion-GAN: pixel-level forward diffusion
- Discriminator: timestep-dependent discriminator `D(y, t)`
- Training: StyleGAN2 non-saturating logistic loss + R1 regularization + path length regularization
- Dataset/config: CIFAR-10, StyleGAN2 CIFAR-like setup, pixel diffusion, `dtarget=0.6`, priority timestep sampling

According to the paper, Diffusion-GAN inherits the architecture and training hyperparameters of the base GAN backbone. For the StyleGAN2-based version, the discriminator is modified to receive the diffusion timestep `t`. Therefore, this notebook directly implements a StyleGAN2-style backbone and adds the Diffusion-GAN forward diffusion and adaptive timestep mechanism on top of it.

In [ ]:
# Install only if needed.
# !pip install torch torchvision tqdm matplotlib

In [ ]:
import os
import math
import random
import copy
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True

## 1. Configuration

In [ ]:
@dataclass
class TrainConfig:
    # Dataset settings
    dataset: str = "CIFAR10"
    data_root: str = "./data"
    resolution: int = 32
    img_channels: int = 3
    num_workers: int = 2

    # StyleGAN2 CIFAR-like model settings
    z_dim: int = 512
    w_dim: int = 512
    mapping_layers: int = 2
    channel_base: int = 16384
    channel_max: int = 512

    # Training settings
    batch_size: int = 64
    total_kimg: int = 25000  # The paper uses 25M real images unless otherwise specified.
    g_lr: float = 0.0025
    d_lr: float = 0.0025
    betas: Tuple[float, float] = (0.0, 0.99)

    # StyleGAN2 lazy regularization settings
    r1_gamma: float = 0.01
    d_reg_interval: int = 16
    pl_weight: float = 2.0
    g_reg_interval: int = 4
    pl_batch_shrink: int = 2
    pl_decay: float = 0.01

    # Exponential moving average for the generator
    ema_kimg: float = 500.0

    # Diffusion-GAN pixel-level diffusion settings
    beta_start: float = 1e-4
    beta_end: float = 2e-2
    t_min: int = 5
    t_max: int = 1000
    noise_sd: float = 0.05
    dtarget: float = 0.6
    ts_dist: str = "priority"  # Either "priority" or "uniform".
    update_T_every: int = 4
    T_step: int = 1
    tepl_size: int = 64
    tepl_zero_count: int = 32

    # Logging and output settings
    outdir: str = "./runs/owncode_diffusion_stylegan2_cifar10"
    sample_every_kimg: int = 100
    save_every_kimg: int = 1000
    print_every: int = 50

cfg = TrainConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Path(cfg.outdir).mkdir(parents=True, exist_ok=True)

print("device:", device)
print("number of CUDA devices:", torch.cuda.device_count())
print(asdict(cfg))

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

## 2. CIFAR-10 Data Loader

In [ ]:
transform = transforms.Compose([
    transforms.Resize(cfg.resolution),
    transforms.CenterCrop(cfg.resolution),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * cfg.img_channels, [0.5] * cfg.img_channels),  # Scale images to [-1, 1].
])

dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=True,
    transform=transform,
    download=True,
)

loader = DataLoader(
    dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
)

def infinite_loader(loader):
    while True:
        for batch in loader:
            yield batch

data_iter = infinite_loader(loader)

real, _ = next(data_iter)
print(real.shape, real.min().item(), real.max().item())

## 3. StyleGAN2 Core Layers

In [ ]:
def normalize_2nd_moment(x, dim=1, eps=1e-8):
    return x * torch.rsqrt(x.square().mean(dim=dim, keepdim=True) + eps)


class FullyConnectedLayer(nn.Module):
    """
    Fully connected layer with equalized learning rate.
    """
    def __init__(
        self,
        in_features,
        out_features,
        bias=True,
        activation="linear",
        lr_multiplier=1.0,
        bias_init=0.0,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.activation = activation
        self.weight = nn.Parameter(torch.randn(out_features, in_features) / lr_multiplier)
        self.bias = nn.Parameter(torch.full([out_features], float(bias_init))) if bias else None
        self.weight_gain = lr_multiplier / math.sqrt(in_features)
        self.bias_gain = lr_multiplier

    def forward(self, x):
        w = self.weight * self.weight_gain
        b = self.bias * self.bias_gain if self.bias is not None else None
        x = F.linear(x, w, b)

        if self.activation == "lrelu":
            x = F.leaky_relu(x, 0.2) * math.sqrt(2)
        elif self.activation == "linear":
            pass
        else:
            raise ValueError(f"Unsupported activation: {self.activation}")

        return x


class Conv2dLayer(nn.Module):
    """
    Convolution layer with equalized learning rate and optional up/downsampling.
    """
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        bias=True,
        activation="lrelu",
        up=False,
        down=False,
    ):
        super().__init__()
        self.up = up
        self.down = down
        self.activation = activation
        self.padding = kernel_size // 2

        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size, kernel_size))
        self.bias = nn.Parameter(torch.zeros(out_channels)) if bias else None
        self.weight_gain = 1 / math.sqrt(in_channels * kernel_size * kernel_size)

    def forward(self, x):
        if self.up:
            x = F.interpolate(x, scale_factor=2, mode="nearest")

        w = self.weight * self.weight_gain
        x = F.conv2d(x, w, bias=self.bias, padding=self.padding)

        if self.down:
            x = F.avg_pool2d(x, kernel_size=2)

        if self.activation == "lrelu":
            x = F.leaky_relu(x, 0.2) * math.sqrt(2)
        elif self.activation == "linear":
            pass
        else:
            raise ValueError(f"Unsupported activation: {self.activation}")

        return x


class MappingNetwork(nn.Module):
    """
    StyleGAN2 mapping network from latent vector z to intermediate latent vector w.
    """
    def __init__(self, z_dim, w_dim, num_layers=2, lr_multiplier=0.01):
        super().__init__()
        layers = []
        in_dim = z_dim

        for _ in range(num_layers):
            layers.append(
                FullyConnectedLayer(
                    in_dim,
                    w_dim,
                    activation="lrelu",
                    lr_multiplier=lr_multiplier,
                )
            )
            in_dim = w_dim

        self.layers = nn.ModuleList(layers)

    def forward(self, z):
        z = normalize_2nd_moment(z)
        x = z

        for layer in self.layers:
            x = layer(x)

        return x


class TimestepMappingNetwork(nn.Module):
    """
    Mapping network used to inject the diffusion timestep into the discriminator.

    In the original StyleGAN2 conditional discriminator, a class label is mapped into
    a conditioning vector. In Diffusion-GAN, the timestep t replaces the class label.
    """
    def __init__(self, t_max=1000, w_dim=512, num_layers=2, lr_multiplier=0.01):
        super().__init__()
        self.t_max = float(t_max)
        self.layers = nn.ModuleList()

        in_dim = 1
        for _ in range(num_layers):
            self.layers.append(
                FullyConnectedLayer(
                    in_dim,
                    w_dim,
                    activation="lrelu",
                    lr_multiplier=lr_multiplier,
                )
            )
            in_dim = w_dim

    def forward(self, t):
        x = t.float().view(-1, 1) / self.t_max

        for layer in self.layers:
            x = layer(x)

        return x

## 4. StyleGAN2 Generator

In [ ]:
class ModulatedConv2d(nn.Module):
    """
    StyleGAN2 modulated convolution with optional demodulation.
    """
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        w_dim,
        demodulate=True,
        up=False,
        eps=1e-8,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.demodulate = demodulate
        self.up = up
        self.eps = eps
        self.padding = kernel_size // 2

        self.weight = nn.Parameter(
            torch.randn(1, out_channels, in_channels, kernel_size, kernel_size)
        )
        self.affine = FullyConnectedLayer(w_dim, in_channels, bias_init=1.0)

    def forward(self, x, w):
        batch, in_ch, height, width = x.shape

        styles = self.affine(w).view(batch, 1, in_ch, 1, 1)
        weight = self.weight * styles

        if self.demodulate:
            demod = torch.rsqrt(weight.square().sum(dim=(2, 3, 4)) + self.eps)
            weight = weight * demod.view(batch, self.out_channels, 1, 1, 1)

        weight = weight.view(
            batch * self.out_channels,
            in_ch,
            self.kernel_size,
            self.kernel_size,
        )

        if self.up:
            x = F.interpolate(x, scale_factor=2, mode="nearest")
            height, width = height * 2, width * 2

        x = x.reshape(1, batch * in_ch, height, width)
        x = F.conv2d(x, weight, padding=self.padding, groups=batch)
        x = x.reshape(batch, self.out_channels, height, width)

        return x


class SynthesisLayer(nn.Module):
    """
    One StyleGAN2 synthesis layer: modulated convolution, noise injection, bias, and leaky ReLU.
    """
    def __init__(self, in_channels, out_channels, w_dim, resolution, kernel_size=3, up=False):
        super().__init__()
        self.resolution = resolution
        self.conv = ModulatedConv2d(in_channels, out_channels, kernel_size, w_dim, up=up)
        self.noise_strength = nn.Parameter(torch.zeros([]))
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(self, x, w, noise_mode="random"):
        x = self.conv(x, w)

        if noise_mode == "random":
            noise = torch.randn([x.shape[0], 1, x.shape[2], x.shape[3]], device=x.device)
            x = x + noise * self.noise_strength
        elif noise_mode == "const":
            # Constant noise is omitted in this compact own-code version.
            # During training, random noise is normally used.
            pass
        elif noise_mode == "none":
            pass
        else:
            raise ValueError(f"Unsupported noise_mode: {noise_mode}")

        x = x + self.bias.view(1, -1, 1, 1)
        x = F.leaky_relu(x, 0.2) * math.sqrt(2)

        return x


class ToRGBLayer(nn.Module):
    """
    StyleGAN2 ToRGB layer.
    """
    def __init__(self, in_channels, out_channels, w_dim):
        super().__init__()
        self.conv = ModulatedConv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            w_dim=w_dim,
            demodulate=False,
        )
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(self, x, w):
        x = self.conv(x, w)
        x = x + self.bias.view(1, -1, 1, 1)
        return x


def num_channels(resolution, channel_base=16384, channel_max=512):
    return min(channel_base // resolution, channel_max)


class SynthesisBlock(nn.Module):
    """
    StyleGAN2 synthesis block for one spatial resolution.
    """
    def __init__(self, in_channels, out_channels, w_dim, resolution, img_channels, is_first):
        super().__init__()
        self.is_first = is_first
        self.resolution = resolution

        if is_first:
            self.const = nn.Parameter(torch.randn(1, out_channels, 4, 4))
            self.conv1 = SynthesisLayer(out_channels, out_channels, w_dim, resolution=4, up=False)
        else:
            self.conv0 = SynthesisLayer(in_channels, out_channels, w_dim, resolution=resolution, up=True)
            self.conv1 = SynthesisLayer(out_channels, out_channels, w_dim, resolution=resolution, up=False)

        self.torgb = ToRGBLayer(out_channels, img_channels, w_dim)

    @property
    def num_conv(self):
        return 1 if self.is_first else 2

    @property
    def num_torgb(self):
        return 1

    def forward(self, x, img, ws, noise_mode="random"):
        # ws has shape [batch, num_conv + num_torgb, w_dim].
        w_iter = iter(ws.unbind(dim=1))

        if self.is_first:
            batch = ws.shape[0]
            x = self.const.repeat(batch, 1, 1, 1)
            x = self.conv1(x, next(w_iter), noise_mode=noise_mode)
        else:
            x = self.conv0(x, next(w_iter), noise_mode=noise_mode)
            x = self.conv1(x, next(w_iter), noise_mode=noise_mode)

        y = self.torgb(x, next(w_iter))

        if img is not None:
            img = F.interpolate(img, scale_factor=2, mode="nearest")
            img = img + y
        else:
            img = y

        return x, img


class Generator(nn.Module):
    """
    StyleGAN2-style generator for CIFAR-10 resolution.
    """
    def __init__(
        self,
        z_dim=512,
        w_dim=512,
        img_resolution=32,
        img_channels=3,
        mapping_layers=2,
        channel_base=16384,
        channel_max=512,
    ):
        super().__init__()
        self.z_dim = z_dim
        self.w_dim = w_dim
        self.img_resolution = img_resolution
        self.img_channels = img_channels

        self.mapping = MappingNetwork(z_dim, w_dim, num_layers=mapping_layers)

        resolutions = [2 ** i for i in range(2, int(math.log2(img_resolution)) + 1)]
        channels = {res: num_channels(res, channel_base, channel_max) for res in resolutions}

        blocks = []
        in_ch = 0
        self.num_ws = 0

        for res in resolutions:
            out_ch = channels[res]
            is_first = res == 4

            block = SynthesisBlock(
                in_channels=in_ch,
                out_channels=out_ch,
                w_dim=w_dim,
                resolution=res,
                img_channels=img_channels,
                is_first=is_first,
            )

            blocks.append((str(res), block))
            self.num_ws += block.num_conv + block.num_torgb
            in_ch = out_ch

        self.blocks = nn.ModuleDict(blocks)

    def forward(self, z, noise_mode="random", return_ws=False):
        w = self.mapping(z)
        ws = w.unsqueeze(1).repeat(1, self.num_ws, 1)

        x = None
        img = None
        w_idx = 0

        for _, block in self.blocks.items():
            block_ws = ws[:, w_idx : w_idx + block.num_conv + block.num_torgb]
            w_idx += block.num_conv + block.num_torgb
            x, img = block(x, img, block_ws, noise_mode=noise_mode)

        img = torch.tanh(img)

        if return_ws:
            return img, ws

        return img

## 5. StyleGAN2 Discriminator with Timestep Conditioning

In [ ]:
class MinibatchStdLayer(nn.Module):
    """
    Minibatch standard deviation layer used in the StyleGAN2 discriminator.
    """
    def __init__(self, group_size=4, num_channels=1):
        super().__init__()
        self.group_size = group_size
        self.num_channels = num_channels

    def forward(self, x):
        N, C, H, W = x.shape
        G = min(self.group_size, N)

        if N % G != 0:
            G = N

        y = x.view(G, -1, self.num_channels, C // self.num_channels, H, W)
        y = y - y.mean(dim=0, keepdim=True)
        y = torch.sqrt(y.square().mean(dim=0) + 1e-8)
        y = y.mean(dim=(2, 3, 4), keepdim=True)
        y = y.mean(dim=2)
        y = y.repeat(G, 1, H, W)

        return torch.cat([x, y], dim=1)


class DiscriminatorBlock(nn.Module):
    """
    Residual discriminator block.
    """
    def __init__(self, in_channels, tmp_channels, out_channels, resolution, img_channels, first_layer=False):
        super().__init__()
        self.resolution = resolution
        self.first_layer = first_layer

        if first_layer:
            self.fromrgb = Conv2dLayer(img_channels, in_channels, kernel_size=1, activation="lrelu")

        self.conv0 = Conv2dLayer(in_channels, tmp_channels, kernel_size=3, activation="lrelu")
        self.conv1 = Conv2dLayer(tmp_channels, out_channels, kernel_size=3, activation="lrelu", down=True)
        self.skip = Conv2dLayer(in_channels, out_channels, kernel_size=1, activation="linear", down=True, bias=False)

    def forward(self, x, img=None):
        if self.first_layer:
            x = self.fromrgb(img)

        skip = self.skip(x)

        x = self.conv0(x)
        x = self.conv1(x)
        x = (x + skip) / math.sqrt(2)

        return x


class DiscriminatorEpilogue(nn.Module):
    """
    Final discriminator block with projection conditioning.
    """
    def __init__(self, in_channels, cmap_dim=512, resolution=4):
        super().__init__()
        self.mbstd = MinibatchStdLayer()
        self.conv = Conv2dLayer(in_channels + 1, in_channels, kernel_size=3, activation="lrelu")
        self.fc = FullyConnectedLayer(in_channels * resolution * resolution, in_channels, activation="lrelu")
        self.out = FullyConnectedLayer(in_channels, cmap_dim, activation="linear")
        self.cmap_dim = cmap_dim

    def forward(self, x, cmap):
        x = self.mbstd(x)
        x = self.conv(x)
        x = x.flatten(1)
        x = self.fc(x)
        x = self.out(x)

        # Projection discriminator: the final logit depends on the timestep conditioning vector.
        logits = (x * cmap).sum(dim=1) / math.sqrt(self.cmap_dim)

        return logits


class Discriminator(nn.Module):
    """
    Diffusion-StyleGAN2 discriminator D(y, t).

    The discriminator follows the StyleGAN2 conditional-discriminator idea,
    but the conditioning variable is the diffusion timestep t instead of a class label.
    """
    def __init__(
        self,
        img_resolution=32,
        img_channels=3,
        channel_base=16384,
        channel_max=512,
        cmap_dim=512,
        t_max=1000,
        mapping_layers=2,
    ):
        super().__init__()
        self.img_resolution = img_resolution
        self.img_channels = img_channels
        self.cmap_dim = cmap_dim

        self.t_mapping = TimestepMappingNetwork(
            t_max=t_max,
            w_dim=cmap_dim,
            num_layers=mapping_layers,
        )

        resolutions = [2 ** i for i in range(int(math.log2(img_resolution)), 2, -1)]
        channels = {res: num_channels(res, channel_base, channel_max) for res in [4] + resolutions}

        blocks = []
        in_ch = channels[img_resolution]

        for idx, res in enumerate(resolutions):
            out_ch = channels[res // 2]

            block = DiscriminatorBlock(
                in_channels=in_ch,
                tmp_channels=in_ch,
                out_channels=out_ch,
                resolution=res,
                img_channels=img_channels,
                first_layer=(idx == 0),
            )

            blocks.append((str(res), block))
            in_ch = out_ch

        self.blocks = nn.ModuleDict(blocks)
        self.epilogue = DiscriminatorEpilogue(in_channels=in_ch, cmap_dim=cmap_dim, resolution=4)

    def forward(self, img, t):
        cmap = self.t_mapping(t)

        x = None
        for _, block in self.blocks.items():
            x = block(x, img if x is None else None)

        logits = self.epilogue(x, cmap)

        return logits

## 6. Diffusion-GAN Forward Diffusion and Adaptive T

In [ ]:
class AdaptiveDiffusion(nn.Module):
    """
    Forward diffusion sampler for Diffusion-GAN.

    It implements:

        q(y | x, t) = N(y; sqrt(alpha_bar_t) x, (1 - alpha_bar_t) sigma^2 I)

    Using reparameterization:

        y = sqrt(alpha_bar_t) x + sqrt(1 - alpha_bar_t) sigma epsilon

    It also maintains the adaptive maximum timestep T and the exploration list t_epl.
    """
    def __init__(
        self,
        beta_start=1e-4,
        beta_end=2e-2,
        t_min=5,
        t_max=1000,
        sigma=0.05,
        dtarget=0.6,
        ts_dist="priority",
        update_step=1,
        tepl_size=64,
        tepl_zero_count=32,
        device="cuda",
    ):
        super().__init__()
        assert ts_dist in ["priority", "uniform"]

        self.t_min = int(t_min)
        self.t_max = int(t_max)
        self.T = int(t_min)
        self.sigma = float(sigma)
        self.dtarget = float(dtarget)
        self.ts_dist = ts_dist
        self.update_step = int(update_step)
        self.tepl_size = int(tepl_size)
        self.tepl_zero_count = int(tepl_zero_count)

        beta = torch.zeros(t_max + 1, device=device)
        beta[1:] = torch.linspace(beta_start, beta_end, t_max, device=device)

        alpha = 1.0 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        alpha_bar[0] = 1.0

        self.register_buffer("beta", beta)
        self.register_buffer("alpha_bar", alpha_bar)

        self.tepl = self.make_tepl().to(device)

    @torch.no_grad()
    def sample_from_p_pi(self, n):
        """
        Sample timesteps from p_pi.

        Uniform sampling:
            P(t) = 1 / T

        Priority sampling:
            P(t) is proportional to t.
        """
        if self.T <= 0:
            return torch.zeros(n, dtype=torch.long)

        if self.ts_dist == "uniform":
            return torch.randint(1, self.T + 1, (n,), dtype=torch.long)

        weights = torch.arange(1, self.T + 1, dtype=torch.float)
        probs = weights / weights.sum()

        return torch.multinomial(probs, n, replacement=True).long() + 1

    @torch.no_grad()
    def make_tepl(self):
        """
        Build the exploration list:

            t_epl = [0, ..., 0, t_1, ..., t_32]

        The default length is 64, with 32 zeros and 32 sampled timesteps.
        """
        n_random = self.tepl_size - self.tepl_zero_count
        zeros = torch.zeros(self.tepl_zero_count, dtype=torch.long)
        sampled = self.sample_from_p_pi(n_random)

        return torch.cat([zeros, sampled], dim=0)

    @torch.no_grad()
    def sample_t(self, batch_size, device):
        """
        Sample timesteps uniformly from the current exploration list t_epl.
        """
        idx = torch.randint(0, self.tepl.numel(), (batch_size,), device=device)
        return self.tepl.to(device)[idx]

    def diffuse(self, x, t):
        """
        Apply the forward diffusion process to a batch of images.

        x must be scaled to [-1, 1].
        """
        a_bar = self.alpha_bar[t].view(-1, 1, 1, 1)
        eps = torch.randn_like(x)
        y = torch.sqrt(a_bar) * x + torch.sqrt(1.0 - a_bar) * self.sigma * eps

        return y

    @torch.no_grad()
    def update_T(self, real_logits):
        """
        Adaptive timestep update.

        The paper defines:

            r_d = E[sign(D(y, t) - 0.5)]
            T = T + sign(r_d - d_target) * C

        This implementation uses logits. Since sigmoid(logit) > 0.5 is equivalent
        to logit > 0, sign(logit) is used.
        """
        rd = torch.sign(real_logits.detach()).float().mean().item()

        direction = 1 if rd > self.dtarget else -1
        self.T += direction * self.update_step
        self.T = int(max(self.t_min, min(self.t_max, self.T)))

        self.tepl = self.make_tepl().to(real_logits.device)

        return rd, self.T

## 7. Loss Functions

In [ ]:
def d_logistic_loss(real_logits, fake_logits):
    """
    Non-saturating logistic discriminator loss.
    """
    return F.softplus(fake_logits).mean() + F.softplus(-real_logits).mean()


def g_nonsaturating_loss(fake_logits):
    """
    Non-saturating generator loss.
    """
    return F.softplus(-fake_logits).mean()


def d_r1_loss(real_logits, real_img):
    """
    R1 gradient penalty on real images.
    """
    grad_real = torch.autograd.grad(
        outputs=real_logits.sum(),
        inputs=real_img,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    return grad_real.square().reshape(real_img.shape[0], -1).sum(1).mean()


def g_path_length_regularize(fake_img, ws, pl_mean, decay=0.01):
    """
    StyleGAN2 path length regularization.
    """
    noise = torch.randn_like(fake_img) / math.sqrt(fake_img.shape[2] * fake_img.shape[3])

    grad = torch.autograd.grad(
        outputs=(fake_img * noise).sum(),
        inputs=ws,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    path_lengths = torch.sqrt(grad.square().sum(2).mean(1) + 1e-8)
    pl_mean_new = pl_mean + decay * (path_lengths.mean() - pl_mean)
    penalty = (path_lengths - pl_mean_new).square().mean()

    return penalty, pl_mean_new.detach(), path_lengths.detach()

## 8. Initialize Models and Optimizers

In [ ]:
G = Generator(
    z_dim=cfg.z_dim,
    w_dim=cfg.w_dim,
    img_resolution=cfg.resolution,
    img_channels=cfg.img_channels,
    mapping_layers=cfg.mapping_layers,
    channel_base=cfg.channel_base,
    channel_max=cfg.channel_max,
).to(device)

D = Discriminator(
    img_resolution=cfg.resolution,
    img_channels=cfg.img_channels,
    channel_base=cfg.channel_base,
    channel_max=cfg.channel_max,
    cmap_dim=cfg.w_dim,
    t_max=cfg.t_max,
    mapping_layers=cfg.mapping_layers,
).to(device)

G_ema = copy.deepcopy(G).eval().requires_grad_(False)

diffusion = AdaptiveDiffusion(
    beta_start=cfg.beta_start,
    beta_end=cfg.beta_end,
    t_min=cfg.t_min,
    t_max=cfg.t_max,
    sigma=cfg.noise_sd,
    dtarget=cfg.dtarget,
    ts_dist=cfg.ts_dist,
    update_step=cfg.T_step,
    tepl_size=cfg.tepl_size,
    tepl_zero_count=cfg.tepl_zero_count,
    device=device,
).to(device)

# Lazy regularization learning-rate compensation.
g_mb_ratio = cfg.g_reg_interval / (cfg.g_reg_interval + 1)
d_mb_ratio = cfg.d_reg_interval / (cfg.d_reg_interval + 1)

opt_G = torch.optim.Adam(
    G.parameters(),
    lr=cfg.g_lr * g_mb_ratio,
    betas=(cfg.betas[0] ** g_mb_ratio, cfg.betas[1] ** g_mb_ratio),
)

opt_D = torch.optim.Adam(
    D.parameters(),
    lr=cfg.d_lr * d_mb_ratio,
    betas=(cfg.betas[0] ** d_mb_ratio, cfg.betas[1] ** d_mb_ratio),
)

pl_mean = torch.zeros([], device=device)

print("G parameters:", sum(p.numel() for p in G.parameters()))
print("D parameters:", sum(p.numel() for p in D.parameters()))
print("G.num_ws:", G.num_ws)
print("Initial T:", diffusion.T)
print("Initial t_epl:", diffusion.tepl[:20].detach().cpu().tolist())

## 9. Sanity Check

In [ ]:
real_img, _ = next(data_iter)
real_img = real_img.to(device)

z = torch.randn(cfg.batch_size, cfg.z_dim, device=device)

with torch.no_grad():
    fake_img, ws = G(z, return_ws=True)
    t = diffusion.sample_t(cfg.batch_size, device)
    y_real = diffusion.diffuse(real_img, t)
    y_fake = diffusion.diffuse(fake_img, t)
    real_logits = D(y_real, t)
    fake_logits = D(y_fake, t)

print("real_img:", real_img.shape)
print("fake_img:", fake_img.shape)
print("ws:", ws.shape)
print("t:", t[:10].detach().cpu().tolist())
print("D logits:", real_logits.shape, fake_logits.shape)

## 10. Sampling and Checkpoint Utilities

In [ ]:
@torch.no_grad()
def update_ema(G_ema, G, batch_size, ema_kimg):
    """
    Update the exponential moving average generator.
    """
    ema_beta = 0.5 ** (batch_size / max(ema_kimg * 1000.0, 1e-8))

    for p_ema, p in zip(G_ema.parameters(), G.parameters()):
        p_ema.copy_(p.lerp(p_ema, ema_beta))

    for b_ema, b in zip(G_ema.buffers(), G.buffers()):
        b_ema.copy_(b)


@torch.no_grad()
def save_samples(G_ema, cur_kimg, n=64, seed=0):
    """
    Generate and save sample images from the EMA generator.
    """
    G_ema.eval()

    gen = torch.Generator(device=device).manual_seed(seed)
    z = torch.randn(n, cfg.z_dim, generator=gen, device=device)

    img = G_ema(z, noise_mode="const").detach().cpu()
    grid = utils.make_grid(img, nrow=8, normalize=True, value_range=(-1, 1))

    out_path = Path(cfg.outdir) / f"samples_kimg_{int(cur_kimg):06d}.png"
    utils.save_image(grid, out_path)

    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis("off")
    plt.title(f"kimg={cur_kimg:.1f}")
    plt.show()

    return out_path


def save_checkpoint(cur_nimg, history):
    """
    Save a training checkpoint.
    """
    ckpt = {
        "G": G.state_dict(),
        "D": D.state_dict(),
        "G_ema": G_ema.state_dict(),
        "opt_G": opt_G.state_dict(),
        "opt_D": opt_D.state_dict(),
        "diffusion_T": diffusion.T,
        "diffusion_tepl": diffusion.tepl.detach().cpu(),
        "pl_mean": pl_mean.detach().cpu(),
        "cur_nimg": cur_nimg,
        "cfg": asdict(cfg),
        "history": history,
    }

    path = Path(cfg.outdir) / f"ckpt_kimg_{cur_nimg // 1000:06d}.pt"
    torch.save(ckpt, path)

    return path

## 11. Training Loop: Diffusion-GAN Algorithm 1 + StyleGAN2 Losses

In [ ]:
total_nimg = cfg.total_kimg * 1000
cur_nimg = 0
step = 0
last_sample_kimg = -1
last_save_kimg = -1

history = {
    "kimg": [],
    "d_loss": [],
    "g_loss": [],
    "r1": [],
    "pl": [],
    "rd": [],
    "T": [],
    "real_score": [],
    "fake_score": [],
}

pbar = tqdm(total=cfg.total_kimg, desc="training kimg")

while cur_nimg < total_nimg:
    step += 1

    # ----------------------------
    # Step I: Update discriminator.
    # ----------------------------
    for p in D.parameters():
        p.requires_grad_(True)

    for p in G.parameters():
        p.requires_grad_(False)

    real_img, _ = next(data_iter)
    real_img = real_img.to(device, non_blocking=True)

    z = torch.randn(cfg.batch_size, cfg.z_dim, device=device)

    with torch.no_grad():
        fake_img = G(z, noise_mode="random")

    t = diffusion.sample_t(cfg.batch_size, device)

    y_real = diffusion.diffuse(real_img, t)
    y_fake = diffusion.diffuse(fake_img, t)

    real_logits = D(y_real, t)
    fake_logits = D(y_fake, t)

    d_loss = d_logistic_loss(real_logits, fake_logits)

    opt_D.zero_grad(set_to_none=True)
    d_loss.backward()
    opt_D.step()

    # Apply R1 regularization lazily.
    r1_val = torch.zeros([], device=device)

    if step % cfg.d_reg_interval == 0:
        real_img, _ = next(data_iter)
        real_img = real_img.to(device, non_blocking=True).requires_grad_(True)

        t_r1 = diffusion.sample_t(cfg.batch_size, device)
        y_real_r1 = diffusion.diffuse(real_img, t_r1)
        real_logits_r1 = D(y_real_r1, t_r1)

        r1_penalty = d_r1_loss(real_logits_r1, real_img)
        r1_loss = r1_penalty * (cfg.r1_gamma * cfg.d_reg_interval / 2)

        opt_D.zero_grad(set_to_none=True)
        r1_loss.backward()
        opt_D.step()

        r1_val = r1_penalty.detach()

    # ----------------------------
    # Step II: Update generator.
    # ----------------------------
    for p in D.parameters():
        p.requires_grad_(False)

    for p in G.parameters():
        p.requires_grad_(True)

    z = torch.randn(cfg.batch_size, cfg.z_dim, device=device)
    fake_img, ws = G(z, noise_mode="random", return_ws=True)

    t_g = diffusion.sample_t(cfg.batch_size, device)
    y_fake_g = diffusion.diffuse(fake_img, t_g)
    fake_logits_g = D(y_fake_g, t_g)

    g_loss = g_nonsaturating_loss(fake_logits_g)

    opt_G.zero_grad(set_to_none=True)
    g_loss.backward()
    opt_G.step()

    # Apply path length regularization lazily.
    pl_val = torch.zeros([], device=device)

    if cfg.pl_weight > 0 and step % cfg.g_reg_interval == 0:
        pl_batch = max(1, cfg.batch_size // cfg.pl_batch_shrink)
        z_pl = torch.randn(pl_batch, cfg.z_dim, device=device)

        fake_pl, ws_pl = G(z_pl, noise_mode="random", return_ws=True)

        pl_penalty, pl_mean, _ = g_path_length_regularize(
            fake_pl,
            ws_pl,
            pl_mean,
            decay=cfg.pl_decay,
        )

        pl_loss = pl_penalty * cfg.pl_weight * cfg.g_reg_interval

        opt_G.zero_grad(set_to_none=True)
        pl_loss.backward()
        opt_G.step()

        pl_val = pl_penalty.detach()

    update_ema(G_ema, G, cfg.batch_size, cfg.ema_kimg)

    # ----------------------------
    # Step III: Update adaptive diffusion.
    # ----------------------------
    rd = None

    if step % cfg.update_T_every == 0:
        rd, T_now = diffusion.update_T(real_logits)
    else:
        T_now = diffusion.T

    cur_nimg += cfg.batch_size
    cur_kimg = cur_nimg / 1000.0

    pbar.n = int(cur_kimg)
    pbar.refresh()

    if step % cfg.print_every == 0:
        with torch.no_grad():
            real_score = real_logits.mean().item()
            fake_score = fake_logits.mean().item()

        history["kimg"].append(cur_kimg)
        history["d_loss"].append(d_loss.item())
        history["g_loss"].append(g_loss.item())
        history["r1"].append(r1_val.item())
        history["pl"].append(pl_val.item())
        history["rd"].append(float(rd) if rd is not None else None)
        history["T"].append(T_now)
        history["real_score"].append(real_score)
        history["fake_score"].append(fake_score)

        pbar.set_description(
            f"kimg {cur_kimg:.1f} | D {d_loss.item():.3f} | G {g_loss.item():.3f} | "
            f"real {real_score:.3f} fake {fake_score:.3f} | T {T_now}"
        )

    if int(cur_kimg) % cfg.sample_every_kimg == 0 and int(cur_kimg) != last_sample_kimg:
        last_sample_kimg = int(cur_kimg)
        save_samples(G_ema, cur_kimg, n=64, seed=42)

    if int(cur_kimg) % cfg.save_every_kimg == 0 and int(cur_kimg) != last_save_kimg:
        last_save_kimg = int(cur_kimg)
        ckpt_path = save_checkpoint(cur_nimg, history)
        print("saved:", ckpt_path)

pbar.close()
print("Training finished.")

## 12. Plot Training Logs

In [ ]:
def plot_history(history):
    """
    Plot loss, adaptive timestep T, and discriminator scores.
    """
    if len(history["kimg"]) == 0:
        print("No logs yet.")
        return

    x = history["kimg"]

    plt.figure(figsize=(7, 4))
    plt.plot(x, history["d_loss"], label="D loss")
    plt.plot(x, history["g_loss"], label="G loss")
    plt.xlabel("kimg")
    plt.ylabel("loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(x, history["T"], label="Adaptive T")
    plt.xlabel("kimg")
    plt.ylabel("T")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(x, history["real_score"], label="D real logit")
    plt.plot(x, history["fake_score"], label="D fake logit")
    plt.xlabel("kimg")
    plt.ylabel("logit")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_history(history)

## 13. Generate Final Samples

In [ ]:
save_samples(G_ema, cur_nimg / 1000.0, n=64, seed=123)

## 14. Suggested Improvement Experiments

Keep the baseline code unchanged first. For each improvement experiment, copy this notebook and change only one component at a time so that the ablation remains interpretable.

Recommended directions:

1. Timestep sampling distribution: compare `priority`, `uniform`, and learned sampling.
2. Noise standard deviation: compare `0.025`, `0.05`, `0.075`, and `0.1`.
3. Beta schedule: compare linear and cosine schedules.
4. Adaptive T update: replace the sign-based rule with an EMA-smoothed `rd`.
5. Timestep embedding: replace scalar normalized timestep mapping with sinusoidal timestep embedding.

For the report, first present this notebook as the baseline implementation, then present one improved notebook as the proposed extension.